# Agente de Itinerario Inteligente

**Projeto Final** — Agentes de IA com LangGraph  
Especializacao em Inteligencia Artificial Aplicada — IFG  

**Equipe:** Eduardo, Israel, Marcelo

---

**O que este agente faz:**  
Recebe um destino de viagem, busca o clima atual via API e gera um roteiro de 1 dia adequado ao clima.  
Se o roteiro nao condizer com o clima, o Validador rejeita e o Planejador refaz (ate 3 tentativas).

```
Pesquisador (clima) -> Planejador (roteiro) -> Validador
                             ^                    |
                             +--- (se rejeitado) -+
```

---
## Secoes 1-3: Marcelo
---

### 1. Instalacao e Imports

Instale as dependencias e importe as bibliotecas necessarias para todo o notebook.

Bibliotecas principais: `langgraph`, `langchain_core`, `langchain_groq`, `requests`

A API key do Groq e necessaria para usar a LLM (secoes 4 e 5). Crie uma gratuitamente em https://console.groq.com/keys e configure como variavel de ambiente:
```python
import os
os.environ["GROQ_API_KEY"] = "gsk_..."
```

### 2. AgentState

Defina o estado compartilhado entre todos os nos do grafo. O AgentState é uma classe que herda de `TypedDict` (da biblioteca `typing`) — funciona como um dicionario tipado que define quais campos existem e seus tipos.

Só é preciso criar uma classe com o nome AgentState. Campos necessarios dentro da classe: `local` (str), `clima_encontrado` (str), `itinerario` (str), `tentativas` (int).

Referencia: https://langchain-ai.github.io/langgraph/concepts/low_level/#state

### 3. Pesquisador (No A)

Recebe `state["local"]` e busca o clima atual usando a API Open-Meteo (gratuita, sem API key).  
Retorna `{"clima_encontrado": "descricao do clima"}`.  

Tarefas:
- Fazer geocoding: converter o nome da cidade (ex: "Rio de Janeiro") em latitude e longitude. A Open-Meteo tem uma API de geocoding pra isso: `https://geocoding-api.open-meteo.com/v1/search?name=Rio+de+Janeiro&count=1`
- Com a lat/lon, buscar clima atual (temperatura + codigo WMO)
- Traduzir codigo WMO para texto legivel (ex: 0 = "Ceu limpo", 61 = "Chuva leve")
- Encapsular como `@tool` do LangChain

Os codigos WMO sao padroes meteorologicos internacionais retornados pela API Open-Meteo no campo `weather_code`. A tabela completa esta na documentacao: https://open-meteo.com/en/docs#weather_variable_documentation (procure por "WMO Weather interpretation codes")

---
## Secoes 4-6: Israel

**Para testar sem depender do Pesquisador, use estes exemplos fixos:**
```python
# Exemplo de state para testar o Planejador:
state_teste = {
    "local": "Rio de Janeiro",
    "clima_encontrado": "Chuva forte, 22°C",
    "itinerario": "",
    "tentativas": 0
}

# Exemplo de state para testar o Validador:
state_validacao = {
    "local": "Rio de Janeiro",
    "clima_encontrado": "Chuva forte, 22°C",
    "itinerario": "9h: Praia de Copacabana, 12h: Trilha Pedra da Gavea",
    "tentativas": 0
}
# (esse roteiro de praia em dia de chuva deve ser REJEITADO pelo Validador)
```
---

### 4. Planejador (No B)

Recebe o clima e gera um roteiro de 1 dia usando a LLM (Groq / Llama 3.3-70b).  
Retorna `{"itinerario": "roteiro gerado"}`.  

Para chamar a LLM, use o `ChatGroq` do LangChain:
```python
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.7)
resposta = llm.invoke("seu prompt aqui")
print(resposta.content)
```
A `temperature` controla a criatividade: 0 = sempre igual, 1 = bem variado. Para roteiros, 0.7 e um bom valor.

Regra de negocio:
- Chuva -> roteiro 100% indoor (museus, shoppings, restaurantes)
- Sol -> roteiro ao ar livre (parques, praias, trilhas)
- Se `tentativas > 0`, o prompt deve pedir para corrigir o roteiro anterior

### 5. Validador (No C)

Verifica se o roteiro gerado pelo Planejador condiz com o clima.  
Se nao condiz, limpa o itinerario para que o Planejador refaca.  
Retorna `{"itinerario": texto_ou_vazio, "tentativas": n+1}`.

Dica: use o `ChatGroq` com `temperature=0` aqui — o Validador precisa ser consistente e objetivo, diferente do Planejador que precisa de criatividade.

### 6. Roteador

Funcao que decide o proximo passo apos o Validador:
- Se o itinerario foi aprovado (nao esta vazio) -> `"fim"`
- Se esgotou as tentativas (>= 3) -> `"fim"`
- Caso contrario -> `"planejador"` (refazer)

---
## Secoes 7-10: Eduardo

**Para testar sem depender das secoes anteriores, use funcoes stub:**
```python
# Stubs para testar o grafo independente:
def pesquisador_stub(state):
    return {"clima_encontrado": "Chuva forte, 22°C"}

def planejador_stub(state):
    if state["tentativas"] == 0:
        return {"itinerario": "9h: Praia de Copacabana"}  # errado de proposito
    return {"itinerario": "9h: Museu Nacional, 12h: Shopping"}  # corrigido

def validador_stub(state):
    if "Praia" in state["itinerario"] and "Chuva" in state["clima_encontrado"]:
        return {"itinerario": "", "tentativas": state["tentativas"] + 1}
    return {"itinerario": state["itinerario"], "tentativas": state["tentativas"] + 1}
```
---

### 7. Montagem do Grafo

Construa o grafo LangGraph:
1. `StateGraph(AgentState)`
2. Adicione os 3 nos
3. Conecte: `START -> pesquisador -> planejador -> validador`
4. Adicione `add_conditional_edges` no validador usando a funcao roteadora
5. Compile com `.compile()`

### 8. Visualizacao do Grafo

Gere a imagem do grafo para incluir na entrega.  
Dica: `grafo.get_graph().draw_mermaid_png()`

### 9. Execucao do Agente

Execute o agente com um destino real e mostre os logs de cada no.  
Dica: use `.stream()` em vez de `.invoke()` para ver o resultado de cada no conforme executa.

### 10. Demonstracao do Ciclo de Rejeicao

Mostre o ciclo funcionando: force um cenario onde o Validador rejeita o roteiro  
(ex: roteiro de praia em dia de chuva) e o Planejador corrige.  
Esse log e um dos entregaveis do projeto.